# <center> <img src="..\img\ITESOLogo.png" alt="ITESO" width="480" height="130"> </center>
# <center> **Departamento de Electrónica, Sistemas e Informática** </center>
---
## <center> **Big Data** </center>
---
### <center> **Spring 2026** </center>
---
### <center> **Lab 04: Data Unions and Joins Pipeline** </center>
---
**Name**: Bryan Edgardo Romo Gonzalez

In [1]:
from spark_utils import SparkUtils

# Create Spark session
su = SparkUtils("Lab04", "spark://spark-master:7077")

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/03/03 02:18:27 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [9]:
!pwd

/opt/spark/work-dir/notebooks/labs


In [13]:
# 1. Agencies
agency_schema = SparkUtils.generate_schema([("agency_id", "int"), ("agency_info", "string")])
df_agencies = su._spark.read.csv("/opt/spark/work-dir/data/car_service/agencies/", schema=agency_schema, header=True)
df_agencies.show()

# 2. Brands
brand_schema = SparkUtils.generate_schema([("brand_id", "int"), ("brand_info", "string")])
df_brands = su._spark.read.csv("/opt/spark/work-dir/data/car_service/brands/", schema=brand_schema, header=True)
df_brands.show()

# 3. Cars
car_schema = SparkUtils.generate_schema([("car_id", "int"), ("car_info", "string")])
df_cars = su._spark.read.csv("/opt/spark/work-dir/data/car_service/cars/", schema=car_schema, header=True)
df_cars.show()

# 4. Customers
customer_schema = SparkUtils.generate_schema([("customer_id", "int"), ("customer_info", "string")])
df_customers = su._spark.read.csv("/opt/spark/work-dir/data/car_service/customers/", schema=customer_schema, header=True)
df_customers.show()

# 5. Rentals
rental_schema = SparkUtils.generate_schema([("rental_id", "int"), ("rental_info", "string")])
df_rentals = su._spark.read.csv("/opt/spark/work-dir/data/car_service/rentals/", schema=rental_schema, header=True)
df_rentals.show()

+---------+--------------------+
|agency_id|         agency_info|
+---------+--------------------+
|        1|{'agency_name': '...|
|        2|{'agency_name': '...|
|        3|{'agency_name': '...|
|        4|{'agency_name': '...|
|        5|{'agency_name': '...|
+---------+--------------------+

+--------+--------------------+
|brand_id|          brand_info|
+--------+--------------------+
|       1|{'brand_name': 'M...|
|       2|{'brand_name': 'B...|
|       3|{'brand_name': 'A...|
|       4|{'brand_name': 'F...|
|       5|{'brand_name': 'B...|
|       6|{'brand_name': 'H...|
|       7|{'brand_name': 'T...|
+--------+--------------------+

+------+--------------------+
|car_id|            car_info|
+------+--------------------+
|     1|{'car_name': 'Cha...|
|     2|{'car_name': 'She...|
|     3|{'car_name': 'Fau...|
|     4|{'car_name': 'Wag...|
|     5|{'car_name': 'Cam...|
|     6|{'car_name': 'Arc...|
|     7|{'car_name': 'Pat...|
|     8|{'car_name': 'Jon...|
|     9|{'car_name'

### Extract a JSON column with get_json_object function

In [27]:
from pyspark.sql.functions import get_json_object

# Agencies
df_json_agencies = df_agencies.withColumn("agency_name", get_json_object("agency_info", "$.agency_name"))
df_json_agencies.show()

# Brands
df_json_brands = df_brands.withColumn("brand_name", get_json_object("brand_info", "$.brand_name"))
df_json_brands.show()

# Cars
df_json_cars = df_cars.withColumn("car_name", get_json_object("car_info", "$.car_name")) \
                      .withColumn("brand_id", get_json_object("car_info", "$.brand_id").cast("int"))
df_json_cars.show()

# Customers
df_json_customers = df_customers.withColumn("customer_name", get_json_object("customer_info", "$.customer_name"))
df_json_customers.show()

# Rentals
df_json_rentals = df_rentals.withColumn("car_id", get_json_object("rental_info", "$.car_id").cast("int")) \
                        .withColumn("customer_id", get_json_object("rental_info", "$.customer_id").cast("int")) \
                        .withColumn("agency_id", get_json_object("rental_info", "$.agency_id").cast("int"))
df_json_rentals.show()

+---------+--------------------+-------------+
|agency_id|         agency_info|  agency_name|
+---------+--------------------+-------------+
|        1|{'agency_name': '...|  NYC Rentals|
|        2|{'agency_name': '...|LA Car Rental|
|        3|{'agency_name': '...| Zapopan Auto|
|        4|{'agency_name': '...|      SF Cars|
|        5|{'agency_name': '...|  Mexico Cars|
+---------+--------------------+-------------+

+--------+--------------------+-------------+
|brand_id|          brand_info|   brand_name|
+--------+--------------------+-------------+
|       1|{'brand_name': 'M...|Mercedes-Benz|
|       2|{'brand_name': 'B...|          BMW|
|       3|{'brand_name': 'A...|         Audi|
|       4|{'brand_name': 'F...|         Ford|
|       5|{'brand_name': 'B...|          BYD|
|       6|{'brand_name': 'H...|        Honda|
|       7|{'brand_name': 'T...|       Toyota|
+--------+--------------------+-------------+

+------+--------------------+--------------------+--------+
|car_id| 

## JOIN

In [29]:
from pyspark.sql.functions import col

final_df = df_json_rentals \
    .join(df_json_cars, on="car_id") \
    .join(df_json_brands, on="brand_id") \
    .join(df_json_agencies, on="agency_id") \
    .join(df_json_customers, on="customer_id") \
    .select("rental_id", "car_name", "agency_name", "customer_name") \
    .orderBy(col("rental_id"), ascending=True) \
    .where(col("rental_id") >= 12740)

final_df.show(5)

[Stage 67:>                                                         (0 + 1) / 2]

+---------+--------------------+-------------+----------------+
|rental_id|            car_name|  agency_name|   customer_name|
+---------+--------------------+-------------+----------------+
|    12740|Villanueva PLC Mo...| Zapopan Auto|    Andrew Gould|
|    12741|  Levy Group Model 9|      SF Cars| Barbara Sanders|
|    12742|Nguyen, Harrell a...|      SF Cars|Katherine Ibarra|
|    12743|Grimes-Green Model 8|LA Car Rental|    Taylor Perry|
|    12744|Jones, Jefferson ...|      SF Cars|  Regina Stewart|
+---------+--------------------+-------------+----------------+
only showing top 5 rows


In [30]:
su._spark.stop()